# B3L — SAM2.1-Large (test 2)

## Why
The Fundamental Review and B3U concluded that the remaining ceiling is **mask quality** (fragments, containers, merged groups), not how evidence is combined. All pipelines so far use **SAM2.1-Small** (~46M parameters); OmniCount used SAM ViT-H (~640M). A larger model is a generic mask-quality lever.

## Pre-declared design (agreed before any B3L result)
| Choice | Rule |
|---|---|
| Model | **SAM2.1-Large** (`sam2.1_hiera_large.pt`, `configs/sam2.1/sam2.1_hiera_l.yaml`), same SAM2 commit and API; checkpoint SHA256 recorded |
| Input | full image (no masking — test 1 covers masking) |
| B0-L | default automatic mask generator, `apply_postprocessing=False` (as frozen B0) |
| Guided-L | the cached `dense_recovery` prompts, one positive point, multimask off |
| Counting | unchanged: GPV_R4 (NMS 0.5 + B0 pool), then H1b on those masks (A3c, depth, D1 unchanged) |
| Reported | B0-L, GPV_R4-L, H1b-L vs B0_v1, GPV_R4_v1, H1b_v1 |
| Targets | T1, T2′, T3′ as in B3S/B3U |

Runs in parallel with `B3I_Masked_Input_SAM2.ipynb` (test 1). If both help, the combination is tested afterwards.

## Run order
T4 → 1.0, 1.1, 1.2 → 2.1, 2.2 → 3.1 (caching, ~2–3 h, resumable) → 4.1 (can run later on CPU after 1.0–1.2).

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3L_SAM2_Large"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3L_SAM2_Large


In [3]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


# 2. SAM2.1-Large

In [4]:
# ============================================================
# Step 2.1 — Install SAM2 (frozen commit)
# ============================================================

import torch
assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
print("SAM2 commit:", subprocess.check_output(["git", "-C", "/content/sam2_repo", "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

/content
/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.1 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
/content
SAM2 commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4
GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0


In [5]:
# ============================================================
# Step 2.2 — Download (once) and build SAM2.1-Large AMG and predictor
# ============================================================

SAM2_L = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_large.pt"
SAM2_L_CFG = "configs/sam2.1/sam2.1_hiera_l.yaml"
if not SAM2_L.exists():
    SAM2_L.parent.mkdir(parents=True, exist_ok=True)
    !wget -q -O "/content/sam2.1_hiera_large.pt" "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
    import shutil; shutil.copy2("/content/sam2.1_hiera_large.pt", SAM2_L)
SAM2_L_SHA = b0.sha256_file(SAM2_L)
print("SAM2.1-Large SHA256:", SAM2_L_SHA, "| size MB:", round(SAM2_L.stat().st_size / 1e6, 1))

amg_L = b0.build_amg(SAM2_L, device=DEVICE, model_cfg=SAM2_L_CFG, verify_sha256=False)
predictor_L = SAM2ImagePredictor(build_sam2(SAM2_L_CFG, str(SAM2_L), device=DEVICE).eval())
print("SAM2.1-Large AMG and predictor ready")

SAM2.1-Large SHA256: 2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318 | size MB: 898.1
SAM2.1-Large AMG and predictor ready


# 3. SAM2.1-Large caching

In [ ]:
# ============================================================
# Step 3.1 — B0-L and guided-L masks on train_dev400 (full image, resumable)
# ============================================================

L_ROOT = TD_ROOT / "sam2_large"
for d in ["b0", "guided", "_done"]:
    (L_ROOT / d).mkdir(parents=True, exist_ok=True)

todo = [r for r in td_df.itertuples(index=False)
        if not (L_ROOT / "_done" / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for i, r in enumerate(tqdm(todo, desc="SAM2.1-Large")):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    img = b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / r.image_id)
    bm, bmeta = b0.run_amg(amg_L, img)
    io_cache.save_b0_masks(L_ROOT / "b0" / f"{int(di):05d}_{stem}.npz", bm, bmeta)
    prompts = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")["prompts_xy"]
    gm, gs = guided_masks_for_prompts(predictor_L, img, prompts)
    io_cache.save_guided_masks(L_ROOT / "guided" / f"{int(di):04d}_{stem}.npz", gm, prompts, gs,
                               img.shape[:2], dataset_index=di, image_id=r.image_id)
    with open(L_ROOT / "_done" / f"{int(di):05d}.json", "w") as f:
        json.dump({"dataset_index": int(di), "b0_L": int(bm.shape[0]), "guided_L": int(gm.shape[0])}, f)
    if (i + 1) % 25 == 0:
        torch.cuda.empty_cache()
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


SAM2.1-Large:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 58.1 min


# 4. Systems and targets

In [ ]:
# ============================================================
# Step 4.1 — B0-L / GPV_R4-L / H1b-L; targets (CPU-capable)
# ============================================================

assert "systems_from_masks" in globals(), "Run Steps 1.0-1.2 first."
L_ROOT = TD_ROOT / "sam2_large"
rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Systems"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    bm, bmeta = io_cache.load_b0_masks(L_ROOT / "b0" / f"{int(di):05d}_{stem}.npz")
    g = io_cache.load_guided_masks(L_ROOT / "guided" / f"{int(di):04d}_{stem}.npz")
    s = systems_from_masks(bm, bmeta["predicted_iou"], g["masks"], g["scores"], a3c, dep, san["target_score"])
    rows.append({"dataset_index": di, **{f"{k}-L": v for k, v in s.items()}})
lg = pd.DataFrame(rows).merge(
    td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
    .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one")

SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1", "B0-L", "GPV_R4-L", "H1b-L"]
res, targets = evaluate(lg, SYS, "SAM2.1-Large, full image (train_dev400)")
print("B0 masks per image, small vs large (means by stratum):")
display(lg.groupby("density_stratum")[["B0_v1", "B0-L"]].mean().reindex(STRATA).round(1))

lg.to_csv(OUT_DIR / "B3L_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3L_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3L_targets.csv", index=False)
with open(OUT_DIR / "B3L_metadata.json", "w") as f:
    json.dump({"notebook": "B3L_SAM2_Large", "test": "2 — SAM2.1-Large, full image",
               "checkpoint": "sam2.1_hiera_large.pt", "config": "configs/sam2.1/sam2.1_hiera_l.yaml",
               "checkpoint_sha256": globals().get("SAM2_L_SHA"),
               "counting": "GPV_R4 (NMS 0.5 + B0 pool), H1b on those masks",
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Systems:   0%|          | 0/400 [00:00<?, ?it/s]

SAM2.1-Large, full image (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,B0-L,33.84,-58.98,6.36,6.82,10.97,19.23,57.32,331.40
4,GPV_R4-L,29.61,-46.73,8.90,7.65,10.95,15.05,41.37,297.87
5,H1b-L,28.57,-40.94,10.70,9.64,11.30,13.97,37.50,276.42


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,B0-L,"+0.80 [-0.17, +1.85]","-0.91 [-1.95, +0.10]","+19.98 [+12.55, +28.22]","+56.42 [+43.18, +70.27]","+5.20 [+3.43, +6.96]",True,False,False,False
1,GPV_R4-L,"+3.34 [+1.75, +5.05]","-0.09 [-1.69, +1.60]","+4.03 [-1.20, +10.55]","+22.88 [+11.22, +35.27]","+0.96 [-0.30, +2.28]",False,False,False,False
2,H1b-L,"+5.14 [+3.05, +7.32]","+1.90 [-0.24, +4.17]","+0.17 [-2.27, +2.57]","+1.43 [-5.25, +8.90]","-0.08 [-0.89, +0.69]",False,True,False,False


B0 masks per image, small vs large (means by stratum):


,B0_v1,B0-L
density_stratum,,
S1_0_20,11.8,14.8
S2_20_40,16.4,19.0
S3_40_60,21.0,24.2
S4_60_80,33.1,38.5
S5_80_95,54.9,56.9
S6_95_100,69.2,75.8


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3L_SAM2_Large


## Section 4 result (test 2)

SAM2.1-Large helps medium-density scenes for every system (S3–S4: B0 13.30 → 10.97, 22.93 → 19.23; GPV_R4 12.45 → 10.95; H1b 12.35 → 11.30) but not S5–S6 (merging unchanged) and adds sparse false masks (B0-L: +3 masks per S1 image). H1b-L: reweighted 28.57 vs 28.65 (−0.08, n.s.); passes T2′, not T1 or T3′.

# 5. Test A — SAM2.1-Large + masked input (M0)

## Pre-declared design
- **Model:** SAM2.1-Large (as Sections 2–4).
- **Masked input:** exactly as B3I variant **M0** — region = pixels where the target wins in the tiled expanded-vocabulary map, no dilation; image × region (outside = black); discard masks with > 50% of pixels outside the region. **M0 only** (in B3I it gave the better reweighted MAE for both GPV_R4 and H1b; one variant halves the run time).
- **Prompts and counting:** unchanged (cached `dense_recovery` prompts; GPV_R4 then H1b).
- **Compared with:** H1b_v1, H1b[M0] (Small, masked; B3I) and H1b-L (Large, full image; Section 4).

**Run order:** T4 → 1.0–1.2 → 2.1–2.2 → 5.1 (~60–70 min, resumable) → 5.2.

In [6]:
# ============================================================
# Step 5.1 — Large + masked input (M0) caching (resumable)
# ============================================================

assert "amg_L" in globals() and "predictor_L" in globals(), "Run Steps 2.1-2.2 first."
TD_TILED = TD_ROOT / "san_vocab_tiled"
LM_ROOT = TD_ROOT / "sam2_large_masked_M0"
for d in ["b0", "guided", "_done"]:
    (LM_ROOT / d).mkdir(parents=True, exist_ok=True)

def keep_inside(masks, region):
    if masks.shape[0] == 0:
        return np.zeros(0, bool)
    inside = (masks & region[None]).reshape(masks.shape[0], -1).sum(1)
    total = masks.reshape(masks.shape[0], -1).sum(1)
    return inside > 0.5 * np.maximum(total, 1)

todo = [r for r in td_df.itertuples(index=False)
        if not (LM_ROOT / "_done" / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for i, r in enumerate(tqdm(todo, desc="Large + masked M0")):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    img = b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / r.image_id)
    with np.load(TD_TILED / f"{stem}.npz") as v:
        reg = v["tiled_argmax"] == 0
    masked = (img * reg[..., None]).astype(np.uint8)
    bm, bmeta = b0.run_amg(amg_L, masked)
    kb = keep_inside(bm, reg)
    io_cache.save_b0_masks(LM_ROOT / "b0" / f"{int(di):05d}_{stem}.npz", bm[kb], filter_meta(bmeta, kb))
    prompts = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")["prompts_xy"]
    gm, gs = guided_masks_for_prompts(predictor_L, masked, prompts)
    kg = keep_inside(gm, reg)
    io_cache.save_guided_masks(LM_ROOT / "guided" / f"{int(di):04d}_{stem}.npz", gm[kg],
                               prompts[kg] if len(prompts) else prompts, gs[kg], img.shape[:2],
                               dataset_index=di, image_id=r.image_id)
    with open(LM_ROOT / "_done" / f"{int(di):05d}.json", "w") as f:
        json.dump({"dataset_index": int(di), "b0_kept": int(kb.sum()), "guided_kept": int(kg.sum())}, f)
    if (i + 1) % 25 == 0:
        torch.cuda.empty_cache()
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


Large + masked M0:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 56.3 min


In [7]:
# ============================================================
# Step 5.2 — Systems and targets (Large + masked M0)
# ============================================================

LM_ROOT = TD_ROOT / "sam2_large_masked_M0"
rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Systems"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    bm, bmeta = io_cache.load_b0_masks(LM_ROOT / "b0" / f"{int(di):05d}_{stem}.npz")
    g = io_cache.load_guided_masks(LM_ROOT / "guided" / f"{int(di):04d}_{stem}.npz")
    s = systems_from_masks(bm, bmeta["predicted_iou"], g["masks"], g["scores"], a3c, dep, san["target_score"])
    rows.append({"dataset_index": di, **{f"{k}-L[M0]": v for k, v in s.items()}})
lm = pd.DataFrame(rows)

prev_L = pd.read_csv(OUT_DIR / "B3L_per_image.csv")[["dataset_index", "B0-L", "GPV_R4-L", "H1b-L"]]
prev_I = pd.read_csv(RESULTS_ROOT / "B3I_Masked_Input_SAM2" / "B3I_per_image.csv")[
    ["dataset_index", "B0[M0]", "GPV_R4[M0]", "H1b[M0]"]]
comb = (lm.merge(prev_L, on="dataset_index").merge(prev_I, on="dataset_index")
        .merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
               .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one"))

SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1", "H1b[M0]", "H1b-L", "B0-L[M0]", "GPV_R4-L[M0]", "H1b-L[M0]"]
res5, targets5 = evaluate(comb, SYS, "Test A — SAM2.1-Large + masked input M0 (train_dev400)")

D5 = OUT_DIR / "section5_large_masked"
D5.mkdir(parents=True, exist_ok=True)
comb.to_csv(D5 / "B3L_s5_per_image.csv", index=False)
res5.to_csv(D5 / "B3L_s5_metrics.csv", index=False)
targets5.to_csv(D5 / "B3L_s5_targets.csv", index=False)
with open(D5 / "B3L_s5_metadata.json", "w") as f:
    json.dump({"section": "5 — test A: SAM2.1-Large + masked input M0",
               "masked_input": "target-winning tiled region, no dilation; discard masks >50% outside",
               "variant_choice": "M0 only (better reweighted MAE than M1 in B3I for GPV_R4 and H1b)",
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", D5)

Systems:   0%|          | 0/400 [00:00<?, ?it/s]

Test A — SAM2.1-Large + masked input M0 (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,H1b[M0],29.58,-49.12,7.41,6.89,12.05,14.83,42.33,299.88
4,H1b-L,28.57,-40.94,10.70,9.64,11.30,13.97,37.50,276.42
5,B0-L[M0],35.09,-64.01,4.06,6.31,10.23,22.50,61.17,345.88
6,GPV_R4-L[M0],30.36,-54.12,5.35,5.28,10.23,16.07,47.28,317.70
7,H1b-L[M0],28.91,-49.22,6.89,6.14,9.93,14.90,42.42,299.43


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,H1b[M0],"+1.85 [-0.08, +3.78]","-0.85 [-2.77, +1.12]","+5.00 [-0.07, +11.45]","+24.90 [+13.83, +37.88]","+0.93 [-0.33, +2.27]",True,False,False,False
1,H1b-L,"+5.14 [+3.05, +7.32]","+1.90 [-0.24, +4.17]","+0.17 [-2.27, +2.57]","+1.43 [-5.25, +8.90]","-0.08 [-0.89, +0.69]",False,True,False,False
2,B0-L[M0],"-1.50 [-2.60, -0.55]","-1.42 [-2.91, +0.10]","+23.83 [+15.73, +32.82]","+70.90 [+57.25, +85.33]","+6.44 [+4.60, +8.35]",True,False,False,False
3,GPV_R4-L[M0],"-0.21 [-1.59, +1.11]","-2.46 [-3.99, -0.89]","+9.95 [+3.22, +18.33]","+42.72 [+29.13, +57.53]","+1.71 [+0.17, +3.37]",True,False,False,False
4,H1b-L[M0],"+1.33 [-0.44, +3.11]","-1.60 [-3.34, +0.16]","+5.08 [-0.08, +11.52]","+24.45 [+12.60, +38.02]","+0.26 [-1.03, +1.64]",True,False,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3L_SAM2_Large/section5_large_masked
